In [43]:
data = {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc"}
selection_string = "resid 618 626 627 628 638 640 642 646 655 657 659 661 665 670 672 674 675 684 686 688 696 698 707 709 711 712 713 714 715 716 717 718 720 725 726 727 729 740 741 743 745 746 747"

In [50]:
data

{'label': [0],
 'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
 'coordinates_filename': '../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc'}

In [45]:
selection_string

'resid 618 626 627 628 638 640 642 646 655 657 659 661 665 670 672 674 675 684 686 688 696 698 707 709 711 712 713 714 715 716 717 718 720 725 726 727 729 740 741 743 745 746 747'

In [46]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch.nn import Embedding, CrossEntropyLoss, Linear, ReLU, Sequential
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, AttentionalAggregation
from torch_geometric.nn import radius_graph
from torch_geometric.utils import scatter
from sklearn.model_selection import train_test_split, KFold

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import json

In [47]:
with open('atom-list.json') as f: 
    atom_type_index = json.load(f)
with open('residue-list.json') as f: 
    residue_index = json.load(f)

In [48]:
def trajectory_to_data(u, selection_string, residue_index, atom_type_index, label):
    for fr in u.trajectory:
        x = torch.from_numpy(u.select_atoms(selection_string).positions)
        residues = list(map(lambda x: residue_index[x], u.select_atoms(selection_string).resnames))
        atom_types = list(map(lambda x: atom_type_index[x], u.select_atoms(selection_string).atoms.types.tolist()))
        d = Data(pos=x, res=torch.tensor(residues, dtype=torch.long), types=torch.tensor(atom_types, dtype=torch.long), label=torch.tensor(label, dtype=torch.long))
        yield d

In [53]:
trj = []    
u = mda.Universe(data['topology_filename'], data['coordinates_filename'])
trj += list(trajectory_to_data(u, selection_string, residue_index, atom_type_index, data['label']))
print(len(u.trajectory)) # to check there are 1500 frames per file

1500


In [56]:
trj[0]

Data(pos=[765, 3], res=[765], types=[765], label=[1])

In [60]:
trj[0].res

tensor([22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 11, 11,
        11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 18,
        18, 18, 18, 18, 18, 18, 18, 18, 18, 18, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 11, 11, 11, 11, 11,
        11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
        11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 22, 22, 22,
        22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 11, 11, 11, 11, 11,
        11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12,
        12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 11, 11, 11,
        11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 21, 21,
        21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21,
        21, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 12, 12, 12,
        12, 12, 12, 12, 12, 12, 12, 12, 

In [61]:
print(np.unique(trj[0].res.numpy()))

[ 2  5  8  9 11 12 13 16 17 18 19 21 22]
